In [13]:
import sys
sys.path.append('..')

import pandas as pd 

import numpy as np 

from src.data_loader import DataLoader 

from src.statistics import StatisticalAnalyzer 
from scipy import stats
import matplotlib.pyplot as plt 

import seaborn as sns 

 

# Load cleaned data 

loader = DataLoader(data_dir="../data") 

df = pd.read_csv(loader.processed_dir / "cleaned_data.csv") 

 

# Initialize analyzer 

analyzer = StatisticalAnalyzer(df) 

all_tests = [] 

print("=" * 60) 

print("HYPOTHESIS TESTING") 

print("=" * 60) 

 

# ==================== 1. T-TESTS ==================== 

print("\n T-TESTS")

if 'flipper_length_mm' in df.columns:
    result = analyzer.t_test('flipper_length_mm', 200)
    print(f"\nOne-sample t-test on flipper_length_mm:")
    print(f"  Mean: {result['mean']:.3f}")
    print(f"  t-statistic: {result['statistic']:.3f}")
    print(f"  p-value: {result['p_value']:.4f}")
    print(f"  Significant: {result['significant']}")
    print(f"  Interpretation: {result['interpretation']}")

    all_tests.append({
        'test': 'One-sample t-test',
        'variable1': 'flipper_length_mm',
        'variable2': 200,
        'statistic': result['statistic'],
        'p_value': result['p_value'],
        'significant': result['significant']
    })

if all(col in df.columns for col in ['body_mass_g', 'species']):
    species_names = df['species'].unique()
    if len(species_names) >= 2:
        group1 = species_names[0]
        group2 = species_names[1]

        data1 = df[df['species'] == group1]['body_mass_g'].dropna()
        data2 = df[df['species'] == group2]['body_mass_g'].dropna()

        statistic, p_value = stats.ttest_ind(data1, data2)

        print(f"\nIndependent t-test: {group1} vs {group2}")
        print(f"  Mean {group1}: {data1.mean():.3f}")
        print(f"  Mean {group2}: {data2.mean():.3f}")
        print(f"  t-statistic: {statistic:.3f}")
        print(f"  p-value: {p_value:.4f}")
        print(f"  Significant: {p_value < 0.05}")

        all_tests.append({
            'test': 'Independent t-test',
            'variable1': f'body_mass_g ({group1})',
            'variable2': f'body_mass_g ({group2})',
            'statistic': statistic,
            'p_value': p_value,
            'significant': p_value < 0.05
        }) 

 

# ==================== 2. ANOVA ==================== 

print("\n ONE-WAY ANOVA")

if all(col in df.columns for col in ['body_mass_g', 'species']):
    groups = []
    for species in df['species'].unique():
        groups.append(df[df['species'] == species]['body_mass_g'].dropna())

    f_statistic, p_value = stats.f_oneway(*groups)

    print(f"\nANOVA: Body Mass by Species")
    print(f"  f-statistic: {f_statistic:.3f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Significant: {p_value < 0.05}")
    if p_value < 0.05:
        print("  Interpretation: At least one species has significantly different body mass")

    all_tests.append({
        'test': 'One-way ANOVA',
        'variable1': 'body_mass_g',
        'variable2': 'species',
        'statistic': f_statistic,
        'p_value': p_value,
        'significant': p_value < 0.05
    })
 

 

# ==================== 3. CHI-SQUARE TEST ==================== 

print("\n CHI-SQUARE TEST")

categorical_cols = df.select_dtypes(include=['object', 'category']).columns

if len(categorical_cols) >= 2:
    col1, col2 = categorical_cols[:2]
    contingency = pd.crosstab(df[col1], df[col2])
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

    print(f"\nChi-square Test: {col1} vs {col2}")
    print(f"  chi2-statistic: {chi2:.3f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  degrees of freedom: {dof}")
    print(f"  Significant: {p_value < 0.05}")
    print(f"  Interpretation: {'Variables are dependent' if p_value < 0.05 else 'Variables are independent'}")

    all_tests.append({
        'test': 'Chi-square Test',
        'variable1': col1,
        'variable2': col2,
        'statistic': chi2,
        'p_value': p_value,
        'significant': p_value < 0.05
    })
 

 

# ==================== 4. SUMMARY OF ALL TESTS ==================== 

print("\n" + "=" * 60)
print("SUMMARY OF HYPOTHESIS TESTS")
print("=" * 60)

summary_df = pd.DataFrame(all_tests)
if not summary_df.empty:
    import os
    os.makedirs('reports', exist_ok=True)   # กันเผื่อโฟลเดอร์ reports ยังไม่มี
    summary_df.to_csv('reports/hypothesis_tests_summary.csv', index=False)
    print("\n Hypothesis test results saved to 'reports/hypothesis_tests_summary.csv'")
    print(summary_df)

HYPOTHESIS TESTING

 T-TESTS

One-sample t-test on flipper_length_mm:
  Mean: 200.892
  t-statistic: 1.180
  p-value: 0.2387
  Significant: False
  Interpretation: Mean not significantly different from 200

Independent t-test: Adelie vs Chinstrap
  Mean Adelie: 3702.961
  Mean Chinstrap: 3733.088
  t-statistic: -0.473
  p-value: 0.6367
  Significant: False

 ONE-WAY ANOVA

ANOVA: Body Mass by Species
  f-statistic: 337.587
  p-value: 0.0000
  Significant: True
  Interpretation: At least one species has significantly different body mass

 CHI-SQUARE TEST

Chi-square Test: species vs island
  chi2-statistic: 299.550
  p-value: 0.0000
  degrees of freedom: 4
  Significant: True
  Interpretation: Variables are dependent

SUMMARY OF HYPOTHESIS TESTS

 Hypothesis test results saved to 'reports/hypothesis_tests_summary.csv'
                 test             variable1                variable2  \
0   One-sample t-test     flipper_length_mm                      200   
1  Independent t-test  body